<a href="https://colab.research.google.com/github/raki-rankawat/thesis-v1/blob/master/Pruning_Rebuild.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1 — mount Drive + install packages
from google.colab import drive
drive.mount('/content/drive')

!pip -q install torch-pruning thop onnx onnxruntime

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 95.1 MB/s eta 0:00:00


In [2]:
# Cell 2 — fixed
import sys, shutil, os

UTILS_SRC = "/content/drive/My Drive/stm32-thesis/utils"

if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/stm32-thesis/utils/")

✅ utils loaded from Drive


In [3]:
# Cell 3 — prepare dataset (same as all your other notebooks)
from utils.dataset import prepare_dataset
from utils.train   import setup_device

device = setup_device(seed=74)
prepare_dataset()

Device: cuda
1/4 Download
⬇️  Downloading VWW archive...
✅ Download complete: /content/vww_work/vw_coco2014_96.tar.gz
2/4 Extract
📦 Extracting VWW archive...
✅ Extraction complete: /content/vww_work/extracted
3/4 Find root
   Root: /content/vww_work/extracted/vw_coco2014_96
4/4 Manifests
✅ Manifests already exist: /content/drive/My Drive/vww_fixed_split_manifests


PosixPath('/content/vww_work/extracted/vw_coco2014_96')

In [4]:
# Cell 4 — fixed import
from utils.structured_rebuild import rebuild_and_measure
rebuild_and_measure(ratios=(0.02, 0.03), ft_epochs=0)

Device: cuda
Train: 7000 | Val: 1500 | Batch: 64
Test: 1500 samples  ⚠️  Use only for final evaluation

DENSE baseline   | params   139,428 | MACs   20,232,040 | test 79.13%
------------------------------------------------------------------------------
STRUCTURED 2%   | params   132,662 (-4.85%) | MACs   19,114,566 (-5.52%)
   accuracy: masked 79.47%  -> rebuilt(pre-FT) 74.73%  -> post-FT 74.73%
   predicted INT8 latency ~ 3.457 ms  (baseline 3.659 ms, board noise 3.659-3.668 ms)
   -> predicted speedup 0.202 ms vs noise 0.009 ms: potentially measurable
------------------------------------------------------------------------------
STRUCTURED 3%   | params   130,006 (-6.76%) | MACs   18,767,143 (-7.24%)
   accuracy: masked 79.67%  -> rebuilt(pre-FT) 75.33%  -> post-FT 75.33%
   predicted INT8 latency ~ 3.394 ms  (baseline 3.659 ms, board noise 3.659-3.668 ms)
   -> predicted speedup 0.265 ms vs noise 0.009 ms: potentially measurable
------------------------------------------------------

In [5]:
# Cell 5 — fine-tune (checkpoints auto-save to Drive)
from utils.structured_rebuild import rebuild_and_measure
rebuild_and_measure(ratios=(0.02, 0.03), ft_epochs=15)

Device: cuda
Train: 7000 | Val: 1500 | Batch: 64
Test: 1500 samples  ⚠️  Use only for final evaluation

DENSE baseline   | params   139,428 | MACs   20,232,040 | test 79.13%
------------------------------------------------------------------------------
Epoch   1/15 | LR 0.000099 | Train 81.37% | Val 78.60% ✅
Epoch   2/15 | LR 0.000096 | Train 81.31% | Val 77.00%
Epoch   3/15 | LR 0.000090 | Train 82.01% | Val 77.07%
Epoch   4/15 | LR 0.000083 | Train 81.67% | Val 76.60%
Epoch   5/15 | LR 0.000075 | Train 81.69% | Val 77.73%
Epoch   6/15 | LR 0.000065 | Train 82.24% | Val 77.53%
Epoch   7/15 | LR 0.000055 | Train 82.80% | Val 77.73%
Epoch   8/15 | LR 0.000045 | Train 82.61% | Val 76.80%
Epoch   9/15 | LR 0.000035 | Train 81.97% | Val 77.20%
Epoch  10/15 | LR 0.000025 | Train 82.67% | Val 77.53%
Epoch  11/15 | LR 0.000017 | Train 83.14% | Val 77.73%
Epoch  12/15 | LR 0.000010 | Train 82.64% | Val 77.87%
Epoch  13/15 | LR 0.000004 | Train 83.06% | Val 77.60%
Epoch  14/15 | LR 0.000001 | T